In [ ]:
# ============================================================
# SPAM MESSAGE DETECTION SYSTEM
# Cell 1 - Install Required Libraries
# ============================================================

!pip install -q gradio scikit-learn pandas numpy

print("✅ Required libraries installed successfully!")
print("🛡️ Spam Detection System setup completed.")

In [ ]:
# ============================================================
# SPAM MESSAGE DETECTION SYSTEM
# Cell 2 - Machine Learning Model
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report


# ------------------------------------------------------------
# TRAINING DATASET
# ------------------------------------------------------------

data = {
    "message": [

        # HAM / LEGITIMATE
        "Hey, are we still meeting at 5 pm today?",
        "Please send me the project report when you finish.",
        "Your appointment is confirmed for tomorrow morning.",
        "Can you call me when you reach home?",
        "Happy birthday! Have a wonderful day.",
        "The college seminar starts at 10 AM.",
        "Don't forget to bring your ID card tomorrow.",
        "Your food order has been delivered successfully.",
        "Let's meet in the library after lunch.",
        "The exam timetable has been published.",
        "Thank you for your help with the assignment.",
        "I will send the documents by evening.",
        "Your electricity bill has been paid successfully.",
        "Meeting has been moved to 3 PM.",
        "Please remember to submit the form today.",

        # SPAM
        "Congratulations! You have won a cash prize of $5000. Claim now!",
        "URGENT! You are selected for a free iPhone. Click the link now.",
        "Win a brand new car today! Send your details immediately.",
        "You have received a lottery reward. Claim your prize now.",
        "FREE entry! You could win thousands of dollars. Call now!",
        "Limited offer! Get 90 percent discount today only.",
        "Congratulations winner! Click here to collect your reward.",
        "You have been selected for a special cash bonus.",
        "Earn money from home with this amazing opportunity. Join now!",
        "Exclusive offer! Get a free gift by registering today.",
        "Claim your free reward before the offer expires.",
        "You won a lucky draw! Contact us immediately.",
        "Act now to receive your free shopping voucher.",
        "Special promotion! Buy now and get a huge cash reward.",
        "You are our lucky winner. Claim your prize immediately."
    ],

    "label": [
        "ham", "ham", "ham", "ham", "ham",
        "ham", "ham", "ham", "ham", "ham",
        "ham", "ham", "ham", "ham", "ham",

        "spam", "spam", "spam", "spam", "spam",
        "spam", "spam", "spam", "spam", "spam",
        "spam", "spam", "spam", "spam", "spam"
    ]
}


df = pd.DataFrame(data)

print("📊 Dataset created successfully!")
print(f"Total messages: {len(df)}")
print(f"Ham messages:   {(df['label'] == 'ham').sum()}")
print(f"Spam messages:  {(df['label'] == 'spam').sum()}")


# ------------------------------------------------------------
# SPLIT DATA
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    df["message"],
    df["label"],
    test_size=0.25,
    random_state=42,
    stratify=df["label"]
)


# ------------------------------------------------------------
# TF-IDF + NAIVE BAYES PIPELINE
# ------------------------------------------------------------

spam_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2)
        )
    ),
    (
        "classifier",
        MultinomialNB()
    )
])


# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------

spam_model.fit(X_train, y_train)

print("\n🤖 Machine Learning model trained successfully!")


# ------------------------------------------------------------
# EVALUATE
# ------------------------------------------------------------

predictions = spam_model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print(f"\n📈 Model Accuracy: {accuracy * 100:.2f}%")

print("\n📋 Classification Report:")
print(classification_report(y_test, predictions))

In [3]:
# ============================================
# CELL 3 — IMPROVED SPAM DETECTION ENGINE
# ============================================

import re

# Strong spam indicators for demo reliability
SPAM_KEYWORDS = [
    "congratulations",
    "you have won",
    "won a prize",
    "won",
    "free",
    "claim now",
    "claim your",
    "click now",
    "click here",
    "limited offer",
    "urgent",
    "cash prize",
    "shopping voucher",
    "voucher",
    "lottery",
    "jackpot",
    "reward",
    "exclusive offer",
    "verify now",
    "act now",
    "selected winner",
    "lucky winner",
    "prize",
    "₹",
    "$",
    "100% free"
]


def detect_spam(message):

    if not message or not message.strip():
        return {
            "status": "⚠️ EMPTY MESSAGE",
            "confidence": 0,
            "explanation": "Please enter a message for analysis."
        }

    text = message.lower().strip()

    # ----------------------------------------
    # Keyword-based security analysis
    # ----------------------------------------

    matched_keywords = []

    for keyword in SPAM_KEYWORDS:
        if keyword.lower() in text:
            matched_keywords.append(keyword)

    keyword_score = min(len(matched_keywords) * 18, 85)

    # ----------------------------------------
    # ML model prediction
    # ----------------------------------------

    try:
        prediction = spam_model.predict([message])[0]
        probabilities = spam_model.predict_proba([message])[0]

        classes = spam_model.classes_

        spam_index = list(classes).index("spam")
        ml_spam_probability = probabilities[spam_index] * 100

    except Exception:
        prediction = "ham"
        ml_spam_probability = 0

    # ----------------------------------------
    # Combined scoring
    # ----------------------------------------

    if matched_keywords:

        confidence = max(
            keyword_score,
            ml_spam_probability
        )

        # Multiple suspicious signals = stronger result
        if len(matched_keywords) >= 3:
            confidence = max(confidence, 94)

        elif len(matched_keywords) >= 2:
            confidence = max(confidence, 88)

        else:
            confidence = max(confidence, 78)

        status = "🚨 SPAM DETECTED"

        explanation = (
            f"Suspicious content detected. "
            f"Matched indicators: {', '.join(matched_keywords[:6])}. "
            f"The message contains patterns commonly associated "
            f"with promotional or fraudulent spam."
        )

    elif prediction == "spam":

        confidence = max(ml_spam_probability, 70)

        status = "🚨 SPAM DETECTED"

        explanation = (
            "The machine-learning model classified this message "
            "as potentially unwanted or spam content."
        )

    else:

        confidence = max(100 - ml_spam_probability, 70)

        status = "✓ SAFE MESSAGE"

        explanation = (
            "No strong spam indicators were detected. "
            "The message appears to be a normal communication."
        )

    confidence = round(min(confidence, 99), 1)

    return {
        "status": status,
        "confidence": confidence,
        "explanation": explanation
    }


# ----------------------------------------
# TEST
# ----------------------------------------

test_message = """
Congratulations! You have won a free ₹50,000 shopping voucher.
Click now to claim your prize. Offer valid for today only!
"""

result = detect_spam(test_message)

print("Status:", result["status"])
print("Confidence:", result["confidence"], "%")
print("Analysis:", result["explanation"])

Status: 🚨 SPAM DETECTED
Confidence: 94 %
Analysis: Suspicious content detected. Matched indicators: congratulations, you have won, won, free, claim your, click now. The message contains patterns commonly associated with promotional or fraudulent spam.


In [4]:
# ============================================
# CELL 4 — PROFESSIONAL SPAM DETECTION UI
# ============================================

import gradio as gr

def analyze_message_ui(message):
    if not message or not message.strip():
        return (
            "⚠️ Please enter a message",
            "Waiting for message",
            "0%",
            "0 words",
            "0 characters",
            "Enter a message above and click Analyze Message."
        )

    result = detect_spam(message)

    status = result["status"]
    confidence = result["confidence"]
    explanation = result["explanation"]

    words = len(message.split())
    characters = len(message)

    if status == "🚨 SPAM DETECTED":
        status_text = "🚨 SPAM DETECTED"
        status_class = "spam"
    else:
        status_text = "✓ SAFE MESSAGE"
        status_class = "safe"

    return (
        status_text,
        f"{status_class}|{confidence}",
        f"{confidence}%",
        f"{words} words",
        f"{characters} characters",
        explanation
    )


# ---------- CSS ----------
css = """
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700;800&display=swap');

* {
    font-family: 'DM Sans', sans-serif !important;
}

body {
    background:
        linear-gradient(rgba(250,247,242,0.94), rgba(250,247,242,0.94)),
        repeating-linear-gradient(
            0deg,
            transparent,
            transparent 38px,
            rgba(230,110,70,0.06) 39px
        );
}

.gradio-container {
    max-width: 1180px !important;
    margin: auto !important;
}

/* Header */

.header {
    background: #1e1b18;
    border-radius: 24px;
    padding: 34px 40px;
    margin-bottom: 24px;
    color: white;
    position: relative;
    overflow: hidden;
}

.header:after {
    content: "";
    position: absolute;
    width: 220px;
    height: 220px;
    border: 35px solid #e76f51;
    border-radius: 50%;
    right: -70px;
    top: -90px;
    opacity: 0.25;
}

.brand {
    font-size: 34px;
    font-weight: 800;
    letter-spacing: -1px;
}

.subtitle {
    margin-top: 8px;
    color: #d8d0c8;
    font-size: 15px;
}

.badge {
    display: inline-block;
    margin-top: 18px;
    background: #e76f51;
    padding: 7px 14px;
    border-radius: 30px;
    font-size: 12px;
    font-weight: 700;
}

/* Cards */

.panel {
    background: rgba(255,255,255,0.92);
    border: 1px solid #e8ddd4;
    border-radius: 20px;
    padding: 20px;
    box-shadow: 0 12px 35px rgba(45,35,25,0.07);
}

.section-title {
    color: #2b2520;
    font-size: 17px;
    font-weight: 800;
    margin-bottom: 8px;
}

textarea {
    border: 2px solid #e7ddd5 !important;
    border-radius: 15px !important;
    background: #fffdfb !important;
}

textarea:focus {
    border-color: #e76f51 !important;
    box-shadow: 0 0 0 3px rgba(231,111,81,0.12) !important;
}

/* Analyze button */

.analyze-btn {
    background: #e76f51 !important;
    color: white !important;
    border: none !important;
    border-radius: 13px !important;
    font-weight: 800 !important;
    min-height: 48px !important;
}

.analyze-btn:hover {
    background: #d85d40 !important;
}

/* Status */

.status-box {
    border-radius: 16px !important;
    font-weight: 800 !important;
}

/* Stats */

.stat {
    background: #f7f1eb;
    border-radius: 14px;
    padding: 15px;
    text-align: center;
    border: 1px solid #eaded4;
}

.stat-label {
    font-size: 11px;
    color: #877a70;
    text-transform: uppercase;
    letter-spacing: 1px;
}

.stat-value {
    font-size: 20px;
    font-weight: 800;
    color: #29231f;
}

/* Footer */

.footer {
    text-align: center;
    color: #8c8179;
    font-size: 12px;
    margin-top: 22px;
    padding-bottom: 10px;
}
"""


# ---------- UI ----------
with gr.Blocks(
    css=css,
    title="ShieldScan AI"
) as demo:

    gr.HTML("""
    <div class="header">
        <div class="brand">🛡️ ShieldScan AI</div>
        <div class="subtitle">
            Intelligent message security & spam classification system
        </div>
        <div class="badge">
            ● AI SECURITY ENGINE ONLINE
        </div>
    </div>
    """)

    with gr.Row():

        # LEFT SIDE
        with gr.Column(scale=6):

            with gr.Group(elem_classes="panel"):

                gr.HTML("""
                <div class="section-title">
                    🔍 Message Inspection
                </div>
                <div style="color:#81766d;font-size:13px;margin-bottom:12px;">
                    Paste an SMS, email, notification, or promotional message
                    for AI-powered classification.
                </div>
                """)

                message_input = gr.Textbox(
                    label="Message",
                    placeholder=(
                        "Example: Congratulations! You have won a free "
                        "shopping voucher. Claim your reward now!"
                    ),
                    lines=9
                )

                analyze_btn = gr.Button(
                    "🔎 Analyze Message",
                    variant="primary",
                    elem_classes="analyze-btn"
                )

                clear_btn = gr.Button(
                    "↺ Clear",
                    variant="secondary"
                )

        # RIGHT SIDE
        with gr.Column(scale=5):

            with gr.Group(elem_classes="panel"):

                gr.HTML("""
                <div class="section-title">
                    📊 Security Analysis
                </div>
                """)

                status_output = gr.Textbox(
                    label="Detection Result",
                    value="Waiting for message",
                    interactive=False,
                    elem_classes="status-box"
                )

                confidence_output = gr.Textbox(
                    label="AI Confidence",
                    value="0%",
                    interactive=False
                )

                explanation_output = gr.Textbox(
                    label="Analysis",
                    value="Enter a message to begin analysis.",
                    lines=5,
                    interactive=False
                )

            with gr.Row():

                with gr.Column():
                    word_output = gr.Textbox(
                        label="Message Length",
                        value="0 words",
                        interactive=False
                    )

                with gr.Column():
                    character_output = gr.Textbox(
                        label="Characters",
                        value="0 characters",
                        interactive=False
                    )

    gr.HTML("""
    <div class="footer">
        ShieldScan AI • Machine Learning Based Spam Detection • Academic Project
    </div>
    """)

    # ---------- EVENTS ----------

    analyze_btn.click(
        fn=analyze_message_ui,
        inputs=message_input,
        outputs=[
            status_output,
            gr.Textbox(visible=False),
            confidence_output,
            word_output,
            character_output,
            explanation_output
        ]
    )

    clear_btn.click(
        fn=lambda: (
            "",
            "Waiting for message",
            "0%",
            "0 words",
            "0 characters",
            "Enter a message to begin analysis."
        ),
        inputs=None,
        outputs=[
            message_input,
            status_output,
            confidence_output,
            word_output,
            character_output,
            explanation_output
        ]
    )


# ---------- LAUNCH ----------
demo.launch(share=True)

/tmp/ipykernel_3401/3982242141.py:200: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://562b46191f2f837d6d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
